# 11) M1.6 Reproducible energy modelling projects — Exercises

**Goals:** record the environment, build scenarios from a base config without side effects, run and compare scenarios through the package, write a property test, and build an audit tool you can run on your own P1 repository.

The setup cell finds the course template and imports its package `hr_energy`. Nothing here modifies the template.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import copy
import math
import re
import shutil
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import pypsa

# find the template whether this notebook runs from exercise/ or exercise/solutions/
TEMPLATE = next(p / "template" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "template" / "pyproject.toml").is_file())
sys.path.insert(0, str(TEMPLATE / "src"))
from hr_energy.config import ConfigError, load_config, validate_config
from hr_energy.model import build_network, solve, summarise

BASE = load_config(TEMPLATE / "config" / "config.yaml")

### Warm-ups

**1. Record the environment**

Return a dict `{package: installed version}` using `importlib.metadata.version` — the list you paste into your README's *Reproduce* section.

In [2]:
def package_versions(names: list[str]) -> dict[str, str]:
    """Installed version of each package."""
    ### BEGIN SOLUTION
    return {name: version(name) for name in names}
    ### END SOLUTION

In [3]:
v = package_versions(["pypsa", "pandas", "pyyaml"])
assert set(v) == {"pypsa", "pandas", "pyyaml"}
assert v["pypsa"] == pypsa.__version__
v

{'pypsa': '1.4.0', 'pandas': '3.0.6', 'pyyaml': '6.0.3'}

**2. Merge changes into a config without touching the original**

Write `deep_update(base, changes)`: return a **new** dict where nested keys in `changes` override `base`, at any depth. The base must stay unchanged — scenario code that silently edits the base config is a classic source of wrong results.

In [4]:
def deep_update(base: dict, changes: dict) -> dict:
    """Recursive merge of `changes` into a deep copy of `base`."""
    ### BEGIN SOLUTION
    out = copy.deepcopy(base)
    for key, value in changes.items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = deep_update(out[key], value)
        else:
            out[key] = copy.deepcopy(value)
    return out
    ### END SOLUTION

In [5]:
merged = deep_update(BASE, {"generators": {"coal": {"p_nom": 0}}})
assert merged["generators"]["coal"]["p_nom"] == 0
assert merged["generators"]["coal"]["marginal_cost"] == 95, "untouched keys survive"
assert BASE["generators"]["coal"]["p_nom"] == 210, "the base config must not change"
assert merged["generators"]["gas_ccgt"] == BASE["generators"]["gas_ccgt"]

### Core

**3. Scenario configs**

Write `scenario_config(base, name, changes)`: apply the changes, set the scenario `name`, and **validate** the result (raising `ConfigError` for anything implausible).

In [6]:
def scenario_config(base: dict, name: str, changes: dict) -> dict:
    """A validated scenario derived from `base`."""
    ### BEGIN SOLUTION
    config = deep_update(base, changes)
    config["name"] = name
    validate_config(config)
    return config
    ### END SOLUTION

In [7]:
no_coal = scenario_config(BASE, "no_coal", {"generators": {"coal": {"p_nom": 0}}})
assert no_coal["name"] == "no_coal" and BASE["name"] == "base"
try:
    scenario_config(BASE, "typo", {"generators": {"hydro_ror": {"p_max_pu": 4.5}}})
    raise AssertionError("an availability of 4.5 must be rejected")
except ConfigError:
    pass

**4. Run and compare scenarios**

Write `run_scenarios(configs)`: build and solve each config with the package, and return a DataFrame indexed by scenario name with `total_cost_EUR`, `mean_price_EUR_per_MWh`, `imports_MWh`, `max_balance_gap_MW` and `cost_check_ok` (use `summarise`). Then compare the base case, a coal phase-out, and cheaper imports.

In [8]:
def run_scenarios(configs: list[dict]) -> pd.DataFrame:
    """One row of results per scenario config."""
    ### BEGIN SOLUTION
    rows = {}
    for config in configs:
        s = summarise(solve(build_network(config), config.get("solver", "highs")))
        rows[config["name"]] = {"total_cost_EUR": s["total_cost_EUR"],
                                "mean_price_EUR_per_MWh": s["mean_price_EUR_per_MWh"],
                                "imports_MWh": s["energy_by_carrier_MWh"].get("imports", 0.0),
                                "max_balance_gap_MW": s["max_balance_gap_MW"],
                                "cost_check_ok": s["cost_check_ok"]}
    return pd.DataFrame.from_dict(rows, orient="index")
    ### END SOLUTION

In [9]:
cheap_imports = scenario_config(BASE, "cheap_imports", {"generators": {"imports": {"marginal_cost": 90}}})
results = run_scenarios([BASE, no_coal, cheap_imports])
assert list(results.index) == ["base", "no_coal", "cheap_imports"]
assert results["cost_check_ok"].all() and (results["max_balance_gap_MW"] < 1e-6).all(), "every run validated"
assert results.at["no_coal", "total_cost_EUR"] > results.at["base", "total_cost_EUR"]
assert results.at["no_coal", "imports_MWh"] > results.at["base", "imports_MWh"], "imports replace coal"
assert results.at["cheap_imports", "mean_price_EUR_per_MWh"] < results.at["base", "mean_price_EUR_per_MWh"]
results.round(1)

               total_cost_EUR  mean_price_EUR_per_MWh  imports_MWh  \
base               19347678.0                   115.9      30495.5   
no_coal            20125580.0                   118.6      55365.7   
cheap_imports      16161662.5                    90.0     170494.2   

               max_balance_gap_MW  cost_check_ok  
base                          0.0           True  
no_coal                       0.0           True  
cheap_imports                 0.0           True  

**5. A property test: more capacity never costs more**

In a cost-minimising dispatch, adding capacity can never *raise* the optimal cost — the optimiser could always leave the new capacity unused. That is a **property** that must hold for any input, which makes it a powerful test. Write `cost_with_extra_capacity(config, generator, extra_MW)` returning `(cost_before, cost_after)`, and check the property for several generators.

In [10]:
def cost_with_extra_capacity(config: dict, generator: str, extra_MW: float) -> tuple[float, float]:
    """Optimal total cost (EUR) before and after adding `extra_MW` to one generator."""
    ### BEGIN SOLUTION
    before = solve(build_network(config)).objective
    bigger = deep_update(config, {"generators": {generator: {
        "p_nom": config["generators"][generator]["p_nom"] + extra_MW}}})
    after = solve(build_network(bigger)).objective
    return float(before), float(after)
    ### END SOLUTION

In [11]:
for gen in ["nuclear_share", "gas_ocgt", "imports"]:
    before, after = cost_with_extra_capacity(BASE, gen, 200)
    assert after <= before + 1e-6, f"adding {gen} capacity raised the cost"
    print(f"+200 MW {gen:13s}: {before:,.0f} -> {after:,.0f} EUR")
before, after = cost_with_extra_capacity(BASE, "nuclear_share", 200)
assert after < before, "cheap capacity that runs must lower the cost"

+200 MW nuclear_share: 19,347,678 -> 15,818,395 EUR


+200 MW gas_ocgt     : 19,347,678 -> 19,347,678 EUR


+200 MW imports      : 19,347,678 -> 19,347,678 EUR


**6. Interpretation**

The OCGT row above shows *no* change at all. Explain why. Then name one bug in a model that the property test would catch but a "results look plausible" check by eye would miss.

<!-- BEGIN SOLUTION -->
The OCGT (160 €/MWh) is more expensive than imports (120 €/MWh), and the system never runs short of capacity in this week, so the optimiser never uses the OCGT — extra OCGT capacity changes nothing, and the cost stays exactly the same. The same holds for imports: they never reach their 1,500 MW limit, so more import capacity is never used either. That is correct behaviour, not a bug. The property test catches bugs where adding capacity *raises* cost, which can only happen if the model is wrong: for example, a sign error that makes capacity a cost in the objective, a constraint that forces new capacity to run (e.g. a minimum output set by mistake), or capacity being added to the wrong generator or bus. Such errors produce results that still look plausible — a few percent higher cost, a slightly different mix — so nobody would spot them by eye, but the property fails immediately.
<!-- END SOLUTION -->

### Challenge

**7. Audit a project**

Write `audit_project(path)` returning a DataFrame with columns `check`, `passed`, `detail` for these checks, with these exact names:

| check | passes when |
|---|---|
| `readme sections` | `README.md` has headings starting with `## Question`, `## Method`, `## Validation`, `## Limitations`, `## Reproduce` |
| `tests` | at least one `tests/test_*.py` |
| `environment` | `pyproject.toml` or `environment.yml` exists |
| `secrets ignored` | `.gitignore` contains a line `.env` |
| `data provenance` | `data/README.md` exists |
| `config` | at least one `config/*.yaml` |
| `no absolute paths` | no `.py` file under `src/` contains `/home/`, `/Users/` or a Windows drive path such as `C:\Users` or `D:/data` (but not URLs like `https://`) |
| `no large files` | no file over 5 MB anywhere in the project (ignore `.git/`) |

Check it on the template (all pass) and on a deliberately broken copy.

In [12]:
REQUIRED_SECTIONS = ["## Question", "## Method", "## Validation", "## Limitations", "## Reproduce"]


def audit_project(path: str | Path) -> pd.DataFrame:
    """Checks every portfolio project must pass before review."""
    ### BEGIN SOLUTION
    root = Path(path)
    rows = []

    def add(check, passed, detail=""):
        rows.append({"check": check, "passed": bool(passed), "detail": detail})

    readme = root / "README.md"
    headings = [l for l in readme.read_text().splitlines() if l.startswith("## ")] if readme.is_file() else []
    missing = [s for s in REQUIRED_SECTIONS if not any(h.startswith(s) for h in headings)]
    add("readme sections", not missing, f"missing: {missing}" if missing else "")
    add("tests", any((root / "tests").glob("test_*.py")))
    add("environment", (root / "pyproject.toml").is_file() or (root / "environment.yml").is_file())
    gi = root / ".gitignore"
    add("secrets ignored", gi.is_file() and ".env" in [l.strip() for l in gi.read_text().splitlines()])
    add("data provenance", (root / "data" / "README.md").is_file())
    add("config", any((root / "config").glob("*.yaml")))
    pattern = re.compile(r"/home/|/Users/|(?<![A-Za-z])[A-Za-z]:[\\/]")
    offenders = [str(p.relative_to(root)) for p in (root / "src").rglob("*.py") if pattern.search(p.read_text())]
    add("no absolute paths", not offenders, ", ".join(offenders))
    big = [str(p.relative_to(root)) for p in root.rglob("*")
           if p.is_file() and ".git" not in p.parts and p.stat().st_size > 5 * 1024 ** 2]
    add("no large files", not big, ", ".join(big))
    return pd.DataFrame(rows)
    ### END SOLUTION

In [13]:
report = audit_project(TEMPLATE).set_index("check")
assert len(report) == 8 and report["passed"].all(), report[~report["passed"]]

with tempfile.TemporaryDirectory() as tmp:
    broken = Path(tmp) / "p1"
    shutil.copytree(TEMPLATE, broken, ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
    gi = broken / ".gitignore"
    gi.write_text(gi.read_text().replace("\n.env\n", "\n"))                       # secret no longer ignored
    readme = broken / "README.md"
    readme.write_text(readme.read_text().replace("## Limitations", "## Notes"))      # limitations removed
    data_py = broken / "src" / "hr_energy" / "data.py"
    data_py.write_text(data_py.read_text() + '\nDATA = "/home/me/Downloads/load.csv"\n')
    bad = audit_project(broken).set_index("check")["passed"]
assert not bad["secrets ignored"] and not bad["readme sections"] and not bad["no absolute paths"]
assert bad.drop(["secrets ignored", "readme sections", "no absolute paths"]).all(), "other checks still pass"
report

                   passed detail
check                           
readme sections      True       
tests                True       
environment          True       
secrets ignored      True       
data provenance      True       
config               True       
no absolute paths    True       
no large files       True       

**8. Passing the audit is not enough**

Your P1 repository will pass this audit long before it is good. List three things a reviewer would look for in P1 that no automated check can verify, and how you would make each one easy to see in your README.

<!-- BEGIN SOLUTION -->
(1) **A clear question with an answer**: the audit checks that a *Question* section exists, not that the question is interesting or answered. Put the answer and the one figure that shows it at the top. (2) **Sound assumptions and honest limitations**: whether imports as a fixed-price generator, synthetic or real load, or a single bus are acceptable for *this* question is a judgement — state each assumption, why it is acceptable, and what result it could change. (3) **Correct interpretation**: tests prove the numbers are internally consistent, not that the explanation of them is right (e.g. claiming solar lowers average prices a lot when it mostly affects midday hours). Interpret results in prose next to the figures, and compare with an outside reference (ENTSO-E actual prices or generation) where you can. Further: data provenance that is actually complete, readable figures with units and sources, and code someone else can follow.
<!-- END SOLUTION -->